In [115]:
import numpy as np
import pandas as pd

import re
from collections import Counter

In [116]:
human_stories_df = pd.read_csv("../data/story_features/human_stories.csv")
ai_stories_df = pd.read_csv("../data/story_features/ai_stories.csv")

In [117]:
def add_sentence_length_features(df: pd.DataFrame):

    sentences_average_lengths = []
    sentences_variances = []
    sentences_max_lengths = []
    sentences_min_lengths = []
    sentences_upper_95 = []
    sentences_lower_5 = []

    for path in df["filepath"]:

        f_raw = open(path)
        story_raw = f_raw.read()

        sentences = re.split(r"[.!?]", story_raw)
        sentence_lengths = [len(i.split()) for i in sentences]

        sentences_average_lengths.append(np.mean(sentence_lengths))
        sentences_variances.append(np.var(sentence_lengths))
        sentences_max_lengths.append(max(sentence_lengths))
        sentences_min_lengths.append(min(sentence_lengths))
        sentences_upper_95.append(np.quantile(sentence_lengths, 0.95))
        sentences_lower_5.append(np.quantile(sentence_lengths, 0.05))


    df.insert(4, "sentence_length_average", sentences_average_lengths)
    df.insert(5, "sentence_length_variance", sentences_variances)
    df.insert(6, "sentence_length_max", sentences_max_lengths)
    df.insert(7, "sentence_length_min", sentences_min_lengths)
    df.insert(8, "sentence_upper_95", sentences_upper_95)
    df.insert(9, "sentence_lower_5", sentences_lower_5)

    return df

In [118]:
def add_sentence_first_word_features(df: pd.DataFrame):

    sentences_unique_first_words = []
    sentences_first_words_diversity = []
    sentences_most_common_first_words_proportion = []

    for path in df["filepath"]:

        f_raw = open(path)
        story_raw = f_raw.read()

        sentences = re.split(r"[.!?]", story_raw)
        starters = []

        for sentence in sentences:
            words = re.findall(r"\b[a-zA-Z]+\b", sentence.lower())

            if words:
                starters.append(words[0])

        counts = Counter(starters)

        total = len(starters)
        unique = len(counts)
        most_common = max(counts.values())

        sentences_unique_first_words.append(unique)
        sentences_first_words_diversity.append(unique / total)
        sentences_most_common_first_words_proportion.append(most_common / total)


    df.insert(10, "sentence_unique_first_words", sentences_unique_first_words)
    df.insert(11, "sentence_first_words_diversity", sentences_first_words_diversity)
    df.insert(12, "sentence_most_common_starter_proportion", sentences_most_common_first_words_proportion)

    return df

In [119]:
def add_word_features(df: pd.DataFrame):

    ttrs = []

    for path in df["filepath"]:

        f_raw = open(path)
        story_raw = f_raw.read()

        words = re.findall(r"\b[a-zA-Z]+\b", story_raw.lower())

        counts = Counter(words)

        total_words = sum(counts.values())
        unique_words = len(counts)

        ttr = unique_words / total_words

        ttrs.append(ttr)

    df.insert(13, "TTR", ttrs)

    return df

In [120]:
#extracting information of every short story
def add_features(df: pd.DataFrame):

    df = add_sentence_length_features(df)
    df = add_sentence_first_word_features(df)
    df = add_word_features(df)

    return df

In [121]:
human_stories_df = add_features(human_stories_df)
ai_stories_df = add_features(ai_stories_df)

In [122]:
human_stories_df.to_csv("../data/story_features/human_stories.csv")
ai_stories_df.to_csv("../data/story_features/ai_stories.csv")